# 05 – Houses only vs. all buildings (task 23)

**Question:** our models learn from *all* buildings, but only 9% of them are houses. Would a model that learns
**only from houses** rank houses better?

- **Version A, all buildings** (as in task 20): train on all buildings of the 4 training folds.
- **Version B, houses only:** train on the houses of the 4 training folds only (about 940 instead of about 10,000).

**Fair comparison:** both versions use the same 5 folds and are judged on exactly the **same validation houses**,
with the same measures (task 17). So any difference comes only from what the model learned from.

**Possible outcomes**
- B better: houses are different enough from other buildings to deserve their own model.
- A better: the extra data from other buildings helps more than it hurts.
- No clear difference: with about 47 houses in each top 20%, one house moves the measure by about 2 points.

Output: `output/tables/task23_houses_only.csv` and `output/tables/task23_per_fold.csv`.

In [5]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge

from config import THRESHOLD
from metrics import evaluate
from cv import summarize, best_alpha
from pipeline import split_xy, build_pipeline

processed = Path("../data/processed")
tables = Path("../output/tables")
tables.mkdir(parents=True, exist_ok=True)

## 1. Load the data and lock away the test set

Exactly as we did in `02_estimate.ipynb`: join the table with the split and keep only the buildings with a fold (0–4).

print

In [6]:
data = pd.read_csv(processed / "building_clean.csv")
split = pd.read_csv(processed / "split.csv")
data = data.merge(split, on="EGID", how="inner", validate="one_to_one")

train = data[data["test"] == False].reset_index(drop=True)   # test set stays closed until task 26
train["fold"] = train["fold"].astype(int)

X, y, is_house, groups = split_xy(train)
folds = train["fold"].to_numpy()


print("Buildings in cross-validation:", len(train), "| of which houses:", is_house.sum())


Buildings in cross-validation: 12710 | of which houses: 1171
  fold 0: training on all buildings = 10168, training on houses only =  937, validation houses = 234
  fold 1: training on all buildings = 10165, training on houses only =  936, validation houses = 235
  fold 2: training on all buildings = 10169, training on houses only =  937, validation houses = 234
  fold 3: training on all buildings = 10169, training on houses only =  937, validation houses = 234
  fold 4: training on all buildings = 10169, training on houses only =  937, validation houses = 234


## 2. One function for both versions

Like `cross_validate` in `cv.py`, with one switch:
- `houses_only=False`: the model learns from all buildings of the training folds (version A).
- `houses_only=True`: the model learns from the houses of the training folds only (version B).

In both cases the model is **judged on the houses of the validation fold only**, so A and B are scored on the same houses.

In [ ]:
def cross_validate_houses(make_model, houses_only):
    rows = []
    for k in np.unique(folds):
        train_part = folds != k                      # the 4 other folds that are for training (≠validation)
        if houses_only:                              # if we look at houses only: 
            train_part = train_part & is_house       # it stays in train_part if its not the validation fold and if its a house
        val_houses = (folds == k) & is_house         # setting validation fold (which is k) and checking if its in k as well as if its a house

        model = make_model()                         # a new, unfitted model in every fold
        model.fit(X[train_part], y[train_part])
        score = model.predict(X[val_houses])         # predicts the scores for the houses in the validation set

        result = evaluate(y[val_houses], score, is_house[val_houses])  #1. actual value of the house 2. our prediction 3. if its a house
        result = result[result["group"] == "houses"]  # all validation buildings are houses here, so one row is enough
        result.insert(0, "fold", k)                  #adds  the fold number at the front (0)
        rows.append(result)
    return pd.concat(rows, ignore_index=True)       # table of one row per fold


def show(per_fold):
    # the main numbers in percent, mean over the 5 folds
    s = summarize(per_fold).iloc[0]
    return {
        "top 20%: above 450 (%)": round(s["share_above_top20_mean"] * 100, 1),
        "± SE": round(s["share_above_top20_se"] * 100, 1),
        "top 10%: above 450 (%)": round(s["share_above_top10_mean"] * 100, 1),
        "top 20%: captured (%)": round(s["captured_top20_mean"] * 100, 1),
        "MAE": round(s["mae_mean"], 1),
    }

## 3. Quick check: is version A the same as in task 20?

Version A with linear regression and the expert rule is exactly the main model of task 20.
Its numbers must match `02_estimate.ipynb`. If they do, the function works.

In [ ]:
lr_all = cross_validate_houses(lambda: build_pipeline(LinearRegression()), houses_only=False)
pd.Series(show(lr_all))   

top 20%: above 450 (%)     54.0
± SE                        3.1
top 10%: above 450 (%)     49.2
top 20%: captured (%)      28.6
MAE                       104.8
dtype: float64

## 4. Linear regression: all buildings vs. houses only

In [12]:
lr_houses = cross_validate_houses(lambda: build_pipeline(LinearRegression()), houses_only=True)

pd.DataFrame({
    "A: trained on all buildings": show(lr_all),
    "B: trained on houses only": show(lr_houses),
}).T

,top 20%: above 450 (%),± SE,top 10%: above 450 (%),top 20%: captured (%),MAE
A: trained on all buildings,54.0,3.1,49.2,28.6,104.8
B: trained on houses only,57.9,2.6,58.8,30.6,108.3


## 5. Ridge: choose alpha separately for each version

With about 940 houses instead of about 10,000 buildings, the model can overfit more easily, so version B may need a
stronger penalty. To be fair, each version gets its **own** best alpha, chosen with the same rule as in task 20
(highest share above 450 in the top 20% of houses; if equal, smaller MAE).

In [ ]:
ALPHAS = np.logspace(-2, 5, 29)   # the same values as for ridge in task 20

def tune(houses_only):
    tables = []
    for alpha in ALPHAS:
        per_fold = cross_validate_houses(lambda: build_pipeline(Ridge(alpha=alpha)), houses_only)
        summary = summarize(per_fold)
        summary.insert(0, "alpha", alpha)
        tables.append(summary)
    return pd.concat(tables, ignore_index=True)

ridge_table = {False: tune(houses_only=False), True: tune(houses_only=True)}
ridge_alpha = {version: best_alpha(table) for version, table in ridge_table.items()}
print("Best alpha, all buildings:", round(ridge_alpha[False], 3))
print("Best alpha, houses only:  ", round(ridge_alpha[True], 3))

ridge_all = cross_validate_houses(lambda: build_pipeline(Ridge(alpha=ridge_alpha[False])), houses_only=False)
ridge_houses = cross_validate_houses(lambda: build_pipeline(Ridge(alpha=ridge_alpha[True])), houses_only=True)

Best alpha, all buildings: 562.341
Best alpha, houses only:   31.623


## 6. Overview

In [ ]:
versions = {
    ("Linear regression", "A: all buildings"): (lr_all, None),
    ("Linear regression", "B: houses only"): (lr_houses, None),
    ("Ridge", "A: all buildings"): (ridge_all, ridge_alpha[False]),
    ("Ridge", "B: houses only"): (ridge_houses, ridge_alpha[True]),
}

overview = pd.DataFrame([
    {"model": model, "trained on": trained_on, "alpha": "–" if alpha is None else f"{alpha:.3g}", **show(per_fold)}
    for (model, trained_on), (per_fold, alpha) in versions.items()
])
overview

               model        trained on alpha  top 20%: above 450 (%)  ± SE  top 10%: above 450 (%)  top 20%: captured (%)    MAE
0  Linear regression  A: all buildings     –                    54.0   3.1                    49.2                   28.6  104.8
1  Linear regression    B: houses only     –                    57.9   2.6                    58.8                   30.6  108.3
2              Ridge  A: all buildings   562                    54.5   3.6                    43.1                   28.7  106.7
3              Ridge    B: houses only  31.6                    60.4   2.0                    54.4                   31.9  107.1


## 7. Fold by fold: in how many folds does "houses only" win?

The mean can hide a lot. Here we compare A and B **in each fold**, on the same houses. If B wins in some folds and
loses in others, the difference is most likely chance.

In [ ]:
def fold_by_fold(per_fold_a, per_fold_b):
    table = pd.DataFrame({
        "A: all buildings (%)": per_fold_a.set_index("fold")["share_above_top20"] * 100,
        "B: houses only (%)": per_fold_b.set_index("fold")["share_above_top20"] * 100,
    })
    table["B minus A (points)"] = table["B: houses only (%)"] - table["A: all buildings (%)"]
    return table.round(1)

per_fold_lr = fold_by_fold(lr_all, lr_houses)
per_fold_ridge = fold_by_fold(ridge_all, ridge_houses)

for name, table in [("Linear regression", per_fold_lr), ("Ridge", per_fold_ridge)]:
    diff = table["B minus A (points)"]
    print(name)
    print(table.to_string())
    print(f"Houses only better in {(diff > 0).sum()} of {len(diff)} folds, worse in {(diff < 0).sum()}, "
          f"equal in {(diff == 0).sum()} | mean difference {diff.mean():+.1f} points "
          f"(± {diff.std() / np.sqrt(len(diff)):.1f} SE)")
    print()

Linear regression
      A: all buildings (%)  B: houses only (%)  B minus A (points)
fold                                                              
0                     46.8                55.3                 8.5
1                     46.8                55.3                 8.5
2                     61.7                55.3                -6.4
3                     57.4                68.1                10.6
4                     57.4                55.3                -2.1
Houses only better in 3 of 5 folds, worse in 2, equal in 0 | mean difference +3.8 points (± 3.4 SE)

Ridge
      A: all buildings (%)  B: houses only (%)  B minus A (points)
fold                                                              
0                     46.8                57.4                10.6
1                     53.2                59.6                 6.4
2                     61.7                59.6                -2.1
3                     63.8                68.1                 4.3
4   

## 8. Save

In [ ]:
overview.to_csv(tables / "task23_houses_only.csv", index=False)
pd.concat({"Linear regression": per_fold_lr, "Ridge": per_fold_ridge}, names=["model"]).to_csv(tables / "task23_per_fold.csv")
print("Saved task23_houses_only.csv and task23_per_fold.csv")

Saved task23_houses_only.csv and task23_per_fold.csv


## 9. What we found

*Draft, check against the numbers above before using it in the report.*

- **Houses only ranks houses better.** Linear regression: 57.9% vs. 54.0% problem houses in the top 20%;
  ridge: 60.4% vs. 54.5%. For the energy office: about 58–60 hits per 100 contacted owners instead of about 54.
- **But the evidence is not strong.** With linear regression, houses only wins in 3 of 5 folds (+3.8 ± 3.4 points),
  which could be chance. With ridge it wins in 4 of 5 folds (+6.0 ± 2.4 points), which is more convincing.
- **The exact values get slightly worse** (MAE about 108 vs. 105): learning from houses only ranks them better
  but is a bit further off in level.
- **Houses-only needs a penalty.** With only about 940 training houses, ridge picks alpha ≈ 32, which holds the model
  back from fitting noise.
- **Caution:** alpha was chosen on the same 5 folds we report, which makes both ridge numbers a bit optimistic
  (for both versions equally). The final word is the test set (task 26).
- **Meaning for the story:** houses seem to behave differently enough from offices and apartment blocks that
  a model focused on them ranks them better, even with about ten times less data.